# Collecting Web Data through an API


*Course:* MGS 4701 W01 — Integrative Application of Business Analytics\
*Session:* Week 2-2 | Thursday, 10 September 2026\
*Instructor:* Dawei Chen

---

### Where this sits

On Tuesday we ranked four routes to web data, in order of preference:

| | Route | |
| --- | --- | --- |
| 1 | An existing dataset | someone already collected and documented it |
| 2 | <span style="color:blue">An official API</span> | <span style="color:blue">today</span> |
| 3 | A documented feed, sitemap or archive | bulk access, no page parsing |
| 4 | <span style="color:orange">Scraping the HTML</span> | <span style="color:orange">Tuesday 15 September — when no API exists</span>|

Today is route 2, and it is the route you should reach for first. Next Tuesday we
do route 4 on university programme pages, where no API exists.

### By the end of this session you will be able to

1. Explain what an API is and how a REST request is put together.
2. Create a GitHub fine-grained personal access token and use it safely.
3. Map each step of a manual collection process onto a line of code.
4. Handle pagination, rate limits, and the three shapes that JSON data arrives in.
5. Save a dataset that someone else could reproduce — the raw response, a stamped table, and a collection log..

### How this notebook works

Cells are marked either **DEMO** — we run it together — or **YOUR TURN** — you
change something and run it yourself. Every YOUR TURN cell ends with an
`assert`(checks that something is true, and stops the program if it isn't) so you can check your own work without waiting for me.

---
# 1. What an API is

**API** = Application Programming Interface  [应用程序编程接口]

- A defined way for one piece of software to request data or services from another.
- You use them constantly without noticing: a weather app pulls from a weather API, "Sign in with Google" is an authentication API, Alipay and Stripe are payment APIs.

### The restaurant analogy

| Restaurant | API |
| --- | --- |
| The ordering system | the API itself |
| The menu page | the **endpoint** (a URL) |
| Submitting your order | the **request** |
| "No coriander, extra rice" | the **parameters** |
| Your membership card | the **token** |

You never enter the kitchen. You ask through a defined channel, and you get back
something predictable.

### For data collection

An API lets you request specific data from a platform in a structured,
predictable, permitted way. Compare that with scraping, where you take the page
a human was meant to read and pull it apart yourself.

There are several kinds of API — REST, GraphQL, SOAP — and several access models
— public, partner, private. **REST is by far the most common for data collection,
and it is what GitHub offers.** That is all you need today.

### How a REST request works

Every request has four parts, and every response has three.

**Request — what you send**

| Part | What it is |
| --- | --- |
| URL | where the request goes |
| Method | `GET` to retrieve data (the only one we use today) |
| Headers | who you are (your token), and what format you want back |
| Parameters | the filters: `?q=data+analyst+interview&sort=stars&page=2` |

**Response — what comes back**

| Part | What it is |
| --- | --- |
| Status code | `200` OK, `401` bad token, `403` forbidden or rate limited, `404` wrong URL, `422` bad query |
| Headers | how much of your rate limit is left |
| Body | the data, as JSON |

In Python, all of it is one line:

```python
response = requests.get(url, headers=headers, params=params)
```

### Anatomy of one URL

```
https://api.github.com/search/repositories?q=data+analyst+interview&per_page=10
└──────── base ───────┘└──── endpoint ───┘└─────────── parameters ───────────┘
```

| Piece | Value | Meaning |
| --- | --- | --- |
| Base URL | `https://api.github.com` | the API's home address |
| Endpoint | `/search/repositories` | which resource you want |
| `q` | `data analyst interview` | what to search for |
| `per_page` | `10` | how many results per page |

### 🖐Try it now: 
Paste that URL into your browser. You will see <span style="color:red">raw JSON</span> — the same thing Python is about to receive.

### The same search, through two doors

You have just sent this query to the API. Now send it as a human.

**Browser** — a page built for a person:
`https://github.com/search?q=data+analyst+interview&type=repositories&s=stars&o=desc`

**API** — the same query, built for a program:
`https://api.github.com/search/repositories?q=data+analyst+interview&sort=stars&order=desc`

Open both in separate tabs. The parameters are the same request in two dialects:

| Browser | API | |
| --- | --- | --- |
| `q=data+analyst+interview` | `q=data+analyst+interview` | identical |
| `type=repositories` | `/search/repositories` | the browser passes it as a parameter; the API puts it in the path |
| `s=stars` | `sort=stars` | |
| `o=desc` | `order=desc` | |

What comes back is not the same kind of thing. 
- The **server** sends <span style="color:red">HTML</span> — a text document of nested tags describing headings, links, avatars and buttons. Your **browser** then renders that text into the page you see. Those are two separate steps, and only the first one exists as far as a program is concerned.
- The **API** skips the whole business. It sends <span style="color:red">JSON</span>: typed fields, no layout, `total_count` as a number you can compute with rather than a figure you read off the screen.
Same data underneath. One is written to be drawn for a person; the other is structured to be read by a program.

🖐**Try this now.** On the browser tab, press Ctrl+U (Cmd+Option+U on a Mac) to see the HTML the server actually sent, then Ctrl+F for a repository name you can see on the page. Is it in there?

- If it is, the page is *static* — everything arrived in the first response.
- If it isn't, the page is *dynamic* — the results were fetched afterwards by JavaScript, and **a simple `requests.get` would come back with nothing useful**.

That test is the first thing we do on Tuesday, and it decides which tool you need.

One more thing. [GitHub's robots.txt](https://github.com/robots.txt) disallows `/search` for automated requests. You may click that link; your program may not. The API is the permitted route — which is rule 9 in a single line.

In [1]:
# DEMO — the four libraries we need
import requests      # sends the HTTP request
import pandas as pd  # holds the result as a table, exports to CSV
import json          # inspects raw JSON
import time          # pauses between requests
from datetime import datetime, timezone
from pathlib import Path

print("Libraries loaded.")

Libraries loaded.


---
# 2. Rate limits — run this *before* you authenticate

An API is someone else's server. They limit how much you can ask for.

GitHub's limit for **unauthenticated** requests is **60 per hour, counted against
the IP address the request came from — not against you.**

Read that again. Everyone in this room is behind the same campus address, so
right now the whole class shares one budget of 60. Let us look at what is left.

In [2]:
# DEMO — our shared, unauthenticated budget
r = requests.get("https://api.github.com/rate_limit", timeout=15)
limits = r.json()["resources"]

print(f"Status: {r.status_code}\n")
for name in ["core", "search"]:
    d = limits[name]
    reset = datetime.fromtimestamp(d["reset"], tz=timezone.utc).strftime("%H:%M:%S UTC")
    print(f"  {name:8}  {d['remaining']:>5} of {d['limit']:>5} remaining   (resets {reset})")

Status: 200

  core         58 of    60 remaining   (resets 06:40:06 UTC)
  search       10 of    10 remaining   (resets 05:41:19 UTC)


In [4]:
r.json()

{'resources': {'code_search': {'limit': 60,
   'remaining': 58,
   'reset': 1789022406,
   'used': 2},
  'core': {'limit': 60, 'remaining': 58, 'reset': 1789022406, 'used': 2},
  'graphql': {'limit': 0, 'remaining': 0, 'reset': 1789022419, 'used': 0},
  'integration_manifest': {'limit': 5000,
   'remaining': 5000,
   'reset': 1789022419,
   'used': 0},
  'search': {'limit': 10, 'remaining': 10, 'reset': 1789018879, 'used': 0}},
 'rate': {'limit': 60, 'remaining': 58, 'reset': 1789022406, 'used': 2}}

Two things to notice.

**`core` shows 60, shared by everyone here.** That is under three requests each
for the whole session. The notebook below would die in its first loop.

**`search` shows 10 — and that is per *minute*.** The search endpoint has its own,
stricter limit, and search is exactly what we are about to use.

A token fixes both, because **authenticated requests are billed to you, not to
the address you happen to be sitting behind.** Your own budget becomes
**5,000 requests an hour** and **30 searches a minute**. The campus network
stops being a shared queue.

---
# 3. Create a fine-grained personal access token

A token is a digital ID card. It tells the server who you are, so it can track
your usage, give you your own allowance, and control what you may reach. Showing
your student card at the library is the same idea.

A **personal access token** acts as a password for programs, but is safer than
one, because it can be **scoped** (only certain actions), **expirable**, and
**revoked** without changing your password.

## Steps — follow along now

1. **github.com** → click your avatar (top right) → **Settings**
2. Scroll to the bottom of the left sidebar → **Developer settings**
3. **Personal access tokens** → **Fine-grained tokens** → **Generate new token**
4. **Token name:** `mgs4701_data_collection`
5. **Expiration:** choose **Custom** and set a date after **17 December 2026**.
   The default 90 days expires on 9 December — six days before your final
   report is due, which is the worst possible moment for authentication to break.
6. **Repository access:** select **Public Repositories (read-only)**
7. **Permissions:** add **none**. You are reading public data; you need nothing
   else. A token that can write to your repositories is a far worse thing to leak
   than one that can only read what is already public.
8. **Generate token**, then **copy it immediately** — GitHub shows it once.

9. Paste it between the quotes in the next cell.

⚠️ **Clear that line again before you push this notebook to your project
repository.** A token typed into a cell is saved inside the `.ipynb` file, so
committing it publishes your credentials. GitHub scans public repositories for
exactly this and will revoke the token automatically — usually within minutes —
which means your notebook stops working the next time you open it.

Two seconds of housekeeping: delete the token, save, then commit.

In [ ]:
# DEMO — authenticate
# Paste your token between the quotes below.
# Clear it again before you push this notebook to your repository.
GITHUB_TOKEN = " "      # <-- paste your token here

GITHUB_TOKEN = GITHUB_TOKEN.strip()

if not GITHUB_TOKEN:
    raise ValueError(
        "GITHUB_TOKEN is empty. Paste your token between the quotes above, "
        "then run this cell again."
    )

HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
    "User-Agent": "MGS4701-teaching-notebook",
}

check = requests.get("https://api.github.com/rate_limit", headers=HEADERS, timeout=15)

if check.status_code == 200:
    res = check.json()["resources"]
    print("Token accepted.")
    print(f"  core   : {res['core']['remaining']:>5} of {res['core']['limit']}")
    print(f"  search : {res['search']['remaining']:>5} of {res['search']['limit']} per minute")
else:
    print(f"Token check FAILED — status {check.status_code}")
    print(check.json().get("message", ""))
    print(f"\nToken length read: {len(GITHUB_TOKEN)} characters "
          "(a fine-grained token is about 90).")
    print("Most likely: you copied only part of it, picked up a stray space "
          "or newline, or the token has expired.")

Token accepted.
  core   :  5000 of 5000
  search :    30 of 30 per minute


`core` should now read close to 5,000 and `search` should read 30. Compare that
with the shared numbers two cells ago. That is the whole argument for
authenticating, and you just watched it happen.

**If your token is not working**, do not sit and wait. Set `USE_CACHED = True`
in the next cell and carry on with data collected earlier — every cell below will
still run. Fix the token afterwards.

In [6]:
# Fallback — only if your token is not working
USE_CACHED = False
CACHE_FILE = Path("cache/github_repos_cached.csv")

if USE_CACHED and CACHE_FILE.exists():
    cached_df = pd.read_csv(CACHE_FILE)
    print(f"Using cached data: {len(cached_df)} rows collected earlier.")
elif USE_CACHED:
    print(f"No cache found at {CACHE_FILE}. Ask for the file.")
else:
    print("Using live API calls.")

Using live API calls.


---
# 4. The manual process, and what replaces it

On Tuesday we imagined twenty assistants collecting job postings by hand. The
same six steps apply here — only the source changes.

**Our question today:** what does the open-source community publish to help
people prepare for analytics job interviews?

Track 3 studies interview content on 牛客网 and Reddit. This is a third source
for the same question — and one you have a personal stake in, since every one of
you will sit an internship or graduate interview this year. GitHub publishes a
well-documented API and the token is free, so it is where we learn the method.
The code transfers unchanged to any other REST API your track needs.

| What a person would do | What the code does | Where |
| --- | --- | --- |
| 1. Choose the source | *you decide this — the program cannot* | your judgement |
| 2. Write the keyword list | builds one query per keyword | a Python list |
| 3. Search, then filter | sends the request with those parameters | `requests.get` |
| 4. Open each result | already in the response | — |
| 5. Copy the fields into a row | pulls named fields into a dictionary | `extract_repo_info()` |
| 6. Repeat, next page | loops the pages | the pagination loop |

Notice row 1. **Choosing what to search for is the one step that never gets
automated.** Everything below is mechanical.

## Which keywords? Do this by hand first

Before writing a query, open github.com in a browser and search each of these.
Write down the result count each time.

- `data analyst interview`
- `data analyst interviews` — plural. Same number?
- `data-analyst-interview` — hyphens instead of spaces
- `business analyst interview` — one word changed

Every one returns a different count. GitHub search is **case-insensitive**, but
**spacing and hyphenation matter**, because a hyphenated phrase can match a topic
tag. Two different things are happening here, and they are not the same:

**Surface variation.** Singular against plural, spaces against hyphens. These
return different totals but heavily overlapping repositories — the same
population seen through a slightly different window.

**Population variation.** "business analyst" against "data analyst" returns a
largely *different* set of repositories, not the same one re-sorted.

Only the second is a decision about what you are studying. Conflating the two is
how a project ends up with a dataset nobody can describe. There is no single
correct keyword, which is why you use several — and why you record which ones.

In [7]:
# DEMO — the scope, decided before any collection
KEYWORDS = [
    "data analyst interview",
    "data analyst interviews",
    "data-analyst-interview",
    "business analyst interview",
    "analytics interview questions",
    "sql interview questions",
]

API_URL   = "https://api.github.com/search/repositories"
PER_PAGE  = 100     # 100 is GitHub's maximum
MAX_PAGES = 3       # most of these keywords run out before page 3
SORT_BY   = "updated"

print(f"{len(KEYWORDS)} keywords, up to {PER_PAGE * MAX_PAGES} results each")
print(f"Maximum records before deduplication: {len(KEYWORDS) * PER_PAGE * MAX_PAGES}")

6 keywords, up to 300 results each
Maximum records before deduplication: 1800


**Why `sort=updated` rather than `sort=stars`?** Sorting by stars returns the
most popular repositories, which are well maintained and have complete
information — every field filled in, every licence declared. That hides the
problem you are here to learn about. Sorting by recent activity returns ordinary
repositories, where fields are missing in the ways real data is missing.

Clean examples teach you nothing about dirty data.

---
# 5. One request, before the loop

Never write the loop first. Make **one** call, look at what comes back, and
confirm it is what you expected. This is the cheapest hour of the whole project.

In [8]:
# DEMO — a single request, five results
params = {
    "q":        KEYWORDS[0],
    "sort":     SORT_BY,
    "order":    "desc",
    "per_page": 5,
    "page":     1,
}

response = requests.get(API_URL, headers=HEADERS, params=params, timeout=20)

print(f"Status code : {response.status_code}")
print(f"URL sent    : {response.url}")
print(f"Searches left this minute: {response.headers.get('x-ratelimit-remaining')}")

data = response.json()
print(f"\ntotal_count : {data['total_count']:,}")
print(f"items returned : {len(data['items'])}")

Status code : 200
URL sent    : https://api.github.com/search/repositories?q=data+analyst+interview&sort=updated&order=desc&per_page=5&page=1
Searches left this minute: 29

total_count : 546
items returned : 5


In [9]:
data

{'total_count': 546,
 'incomplete_results': False,
 'items': [{'id': 1297945531,
   'node_id': 'R_kgDOTV0Tuw',
   'name': 'tableau-60-days-learning-challenge',
   'full_name': 'vishalsvnair/tableau-60-days-learning-challenge',
   'private': False,
   'owner': {'login': 'vishalsvnair',
    'id': 119663207,
    'node_id': 'U_kgDOByHqZw',
    'avatar_url': 'https://avatars.githubusercontent.com/u/119663207?v=4',
    'gravatar_id': '',
    'url': 'https://api.github.com/users/vishalsvnair',
    'html_url': 'https://github.com/vishalsvnair',
    'followers_url': 'https://api.github.com/users/vishalsvnair/followers',
    'following_url': 'https://api.github.com/users/vishalsvnair/following{/other_user}',
    'gists_url': 'https://api.github.com/users/vishalsvnair/gists{/gist_id}',
    'starred_url': 'https://api.github.com/users/vishalsvnair/starred{/owner}{/repo}',
    'subscriptions_url': 'https://api.github.com/users/vishalsvnair/subscriptions',
    'organizations_url': 'https://api.githu

## Look at those last two numbers

`total_count` is in the hundreds or thousands. `items` has five.

**A successful request does not mean you have the data.** The API told you how
many matches exist and then handed you one page of them. On Tuesday this was the
first verification question — *did it stop at page one?* — and here it is, on
your own screen, on your first call.

You get the rest through **pagination**, which we come to in section 9. Note now
that GitHub's search endpoint will not return more than **1,000 results per
query**, however many pages you ask for.

That cap bites unevenly here. Most of our keywords match a few hundred
repositories, so we can collect the entire population — an unusual luxury, and
worth noticing when it happens. But `sql interview questions` matches close to
two thousand, so for that one we can only ever see half. Needing more means
splitting the query — by date range, by language — not asking harder.

---
# 🖐 YOUR TURN — Task 1 (8 minutes)

Run one search with **your own keyword**. Something from your project track, a
tool you have heard of, anything you are curious about.

Then report three things: `total_count`, how many items came back, and the top
result.

**Compare with the person next to you.** Your totals will differ by orders of
magnitude. The number of items returned will be identical. That difference is
the point.

In [10]:
# YOUR TURN — change MY_KEYWORD, then run
MY_KEYWORD = "power bi interview"     # <-- change this

my_params = {
    "q":        MY_KEYWORD,
    "sort":     "updated",
    "order":    "desc",
    "per_page": 10,          # <-- try changing this too
    "page":     1,
}

my_response = requests.get(API_URL, headers=HEADERS, params=my_params, timeout=20)
my_data = my_response.json()
my_items = my_data.get("items", [])

print(f"Keyword      : {MY_KEYWORD}")
print(f"Status       : {my_response.status_code}")
print(f"total_count  : {my_data.get('total_count', 0):,}")
print(f"items back   : {len(my_items)}")
if my_items:
    top = max(my_items, key=lambda x: x["stargazers_count"])
    print(f"Most starred : {top['full_name']}  ({top['stargazers_count']:,} stars)")

# --- self-check ---
assert my_response.status_code == 200, f"Request failed with {my_response.status_code}"
assert len(my_items) > 0, "No results — try a broader keyword"
assert my_data["total_count"] >= len(my_items), "total_count should be at least the page size"
print("\n✓ Checks passed.")

Keyword      : power bi interview
Status       : 200
total_count  : 209
items back   : 10
Most starred : Jaakuice/SWC_Big_Brain_Ai  (1 stars)

✓ Checks passed.


**Stretch:** add a qualifier to your query and watch `total_count` move —
`"data analyst interview created:>2024-01-01"` or
`"data analyst interview language:Python"`.
These are GitHub search qualifiers, documented under *Searching for repositories*
in the GitHub docs. Filtering at the source is always cheaper than filtering
afterwards.

---
# 7. Reading the JSON — three shapes

The response is a Python dictionary once you call `.json()`. Inside `items`,
each repository is another dictionary. Its values come in three shapes, and each
one breaks in a different way.

In [11]:
# DEMO — one repository, raw
repo = data["items"][0]
print(json.dumps(repo, indent=2, ensure_ascii=False)[:1200])
print("\n... (truncated — a single repository has over 80 fields)")

{
  "id": 1297945531,
  "node_id": "R_kgDOTV0Tuw",
  "name": "tableau-60-days-learning-challenge",
  "full_name": "vishalsvnair/tableau-60-days-learning-challenge",
  "private": false,
  "owner": {
    "login": "vishalsvnair",
    "id": 119663207,
    "node_id": "U_kgDOByHqZw",
    "avatar_url": "https://avatars.githubusercontent.com/u/119663207?v=4",
    "gravatar_id": "",
    "url": "https://api.github.com/users/vishalsvnair",
    "html_url": "https://github.com/vishalsvnair",
    "followers_url": "https://api.github.com/users/vishalsvnair/followers",
    "following_url": "https://api.github.com/users/vishalsvnair/following{/other_user}",
    "gists_url": "https://api.github.com/users/vishalsvnair/gists{/gist_id}",
    "starred_url": "https://api.github.com/users/vishalsvnair/starred{/owner}{/repo}",
    "subscriptions_url": "https://api.github.com/users/vishalsvnair/subscriptions",
    "organizations_url": "https://api.github.com/users/vishalsvnair/orgs",
    "repos_url": "https://a

In [12]:
# DEMO — the three shapes
repo = data["items"][0]

print("1. FLAT — a value sits directly under a key")
print(f"   full_name        : {repo['full_name']}")
print(f"   stargazers_count : {repo['stargazers_count']}")

print("\n2. NESTED — a dictionary inside the dictionary")
print(f"   owner.login      : {repo['owner']['login']}")
print(f"   owner.type       : {repo['owner']['type']}")

print("\n3. LIST — several values under one key")
print(f"   topics           : {repo['topics']}")

1. FLAT — a value sits directly under a key
   full_name        : vishalsvnair/tableau-60-days-learning-challenge
   stargazers_count : 0

2. NESTED — a dictionary inside the dictionary
   owner.login      : vishalsvnair
   owner.type       : User

3. LIST — several values under one key
   topics           : []


## Where each shape goes wrong

**Nested values disappear when they are `null`.** A repository with no licence
has `"license": null`, so `repo["license"]["name"]` raises
`TypeError: 'NoneType' object is not subscriptable`.

**And `.get()` does not save you.** This is the trap:

```python
repo.get("description", "")     # returns None, NOT ""
```

`.get(key, default)` returns the default only when the **key is missing**. When
the key is present and its value is `null`, you get `None` back. The reliable
idiom is:

```python
(repo.get("description") or "")   # None becomes ""
```

**Lists do not belong in a spreadsheet cell.** `topics` arrives as
`['ai', 'agents']`. Put that in a DataFrame and every aggregation you try will
fail. Join it into a string now, split it later if you need to.

Let us see how often this actually happens.

In [13]:
# DEMO — how messy is it, really?
sample = data["items"]
n = len(sample)

print(f"Out of {n} repositories in this page:")
print(f"  description is null : {sum(1 for r in sample if r.get('description') is None)}")
print(f"  license is null     : {sum(1 for r in sample if r.get('license') is None)}")
print(f"  language is null    : {sum(1 for r in sample if r.get('language') is None)}")
print(f"  topics is empty     : {sum(1 for r in sample if not r.get('topics'))}")

Out of 5 repositories in this page:
  description is null : 0
  license is null     : 5
  language is null    : 3
  topics is empty     : 5


Expect something startling on this topic. The great majority of these
repositories carry **no licence at all**, most declare no topics, and many have
no primary language, because they are collections of notes and questions rather
than code.

**These are findings, not defects.** "Most interview-preparation repositories are
published with no licence" is a result about how this material circulates — shared
freely and informally, with nobody thinking about reuse rights. On Tuesday the
equivalent was salary listed as *not stated*. The missingness is data. Report it
in your methods section; do not quietly fill it in.

---
# 8. One repository into one row

Step 5 of the manual process — *copy the fields into a row* — becomes a function.
Written once, called for every repository across every keyword and page.

Each field is chosen because it measures something, not because it was available:

| Field | What it stands for |
| --- | --- |
| `full_name` | identity |
| `description` | what it does — text for later analysis |
| `stargazers_count` | attention |
| `forks_count` | reuse |
| `open_issues_count` | engagement |
| `created_at` | when it entered |
| `updated_at` | whether it is still alive |
| `language` | implementation |
| `license_name` | commercial openness |
| `owner_type` | individual or organisation |
| `topics` | self-declared category |

**A note on `owner_login`.** That is a person's account name, and on this topic
almost every repository belongs to an individual rather than an organisation, so
the question is live rather than theoretical. Tuesday's rule 6
says collect the posting, not the poster. A GitHub account that published a
public repository is closer to an author than to a job applicant, and
`owner_type` (User or Organization) carries most of the analytical value without
naming anyone. We keep `owner_type` and leave `owner_login` out. If your project
needs the account name, be able to say why.

In [14]:
# DEMO — extraction, with every trap handled
def extract_repo_info(repo, keyword):
    """Turn one repository's JSON into one flat row."""
    licence = repo.get("license") or {}       # null -> {}
    owner   = repo.get("owner") or {}
    return {
        "full_name":         repo.get("full_name") or "",
        "description":       repo.get("description") or "",     # None -> ""
        "stargazers_count":  repo.get("stargazers_count", 0),
        "forks_count":       repo.get("forks_count", 0),
        "open_issues_count": repo.get("open_issues_count", 0),
        "created_at":        repo.get("created_at") or "",
        "updated_at":        repo.get("updated_at") or "",
        "language":          repo.get("language") or "",
        "license_name":      licence.get("name") or "",          # nested + null
        "owner_type":        owner.get("type") or "",
        "topics":            "; ".join(repo.get("topics") or []),  # list -> string
        "html_url":          repo.get("html_url") or "",           # provenance
        "search_keyword":    keyword,                              # provenance
    }


example = extract_repo_info(data["items"][0], KEYWORDS[0])
for k, v in example.items():
    print(f"  {k:20} {str(v)[:60]}")

  full_name            vishalsvnair/tableau-60-days-learning-challenge
  description          📊 A 60-day Tableau learning challenge covering fundamentals,
  stargazers_count     0
  forks_count          0
  open_issues_count    0
  created_at           2026-07-12T03:22:22Z
  updated_at           2026-09-10T03:15:21Z
  language             
  license_name         
  owner_type           User
  topics               
  html_url             https://github.com/vishalsvnair/tableau-60-days-learning-cha
  search_keyword       data analyst interview


Every line in that function defends against something real: `or {}` for a null
licence, `or ""` for a null description, `"; ".join(...)` for the topics list.

The last two columns are provenance. `html_url` records where each row came
from; `search_keyword` records how it was found. Neither can be reconstructed
afterwards, and both are required for A1.

---
# 9. Pagination and politeness

APIs do not hand over everything at once. They return **pages**. To collect more
than one page you change the `page` parameter and ask again.

```
page=1  →  100 results  →  keep
page=2  →  100 results  →  keep
page=3  →    0 results  →  stop
```

**Stop when a page comes back empty**, never at a number you guessed.

Two rules from Tuesday apply directly. **Rule 2** — pause between requests. One
second is enough here, and it keeps you well inside the 30-searches-a-minute
allowance. **Rule 9** — when the API tells you it is unhappy, listen: a `403`
means slow down, not try harder.

In [ ]:
# DEMO — one page, with the failure paths handled
RAW_PAGES = []      # every response kept exactly as it arrived — rule 7


def fetch_page(keyword, page, per_page=PER_PAGE, max_retries=3):
    """Fetch one page. Retries the SAME page on rate limiting."""
    params = {"q": keyword, "sort": SORT_BY, "order": "desc",
              "per_page": per_page, "page": page}

    for attempt in range(1, max_retries + 1):
        r = requests.get(API_URL, headers=HEADERS, params=params, timeout=20)

        if r.status_code == 200:
            payload = r.json()
            RAW_PAGES.append({"keyword": keyword, "page": page,
                              "url": r.url, "payload": payload})
            return payload.get("items", [])

        if r.status_code in (403, 429):
            wait = 20 * attempt
            print(f"    rate limited on page {page}; waiting {wait}s "
                  f"(attempt {attempt} of {max_retries})")
            time.sleep(wait)
            continue                      # retries the SAME page

        print(f"    page {page} failed with {r.status_code}: "
              f"{r.json().get('message', '')}")
        return None                       # None means "failed", not "empty"

    print(f"    page {page} gave up after {max_retries} attempts")
    return None


test_items = fetch_page(KEYWORDS[0], page=1, per_page=5)
print(f"Returned {len(test_items) if test_items is not None else 'FAILED'} items")

Note what `fetch_page` returns: a **list** on success, an **empty list** when a
page genuinely has no more results, and **`None`** when the request failed. Those
are three different situations and collapsing them is how you end up reporting a
complete dataset that quietly lost two hundred rows.

In [ ]:
# DEMO — the full collection loop
all_rows = []
failures = []

for keyword in KEYWORDS:
    print(f"Searching: {keyword!r}")
    for page in range(1, MAX_PAGES + 1):
        items = fetch_page(keyword, page)

        if items is None:                 # failed
            failures.append((keyword, page))
            break
        if len(items) == 0:               # genuinely exhausted
            print(f"    page {page}: empty, moving on")
            break

        all_rows.extend(extract_repo_info(r, keyword) for r in items)
        print(f"    page {page}: {len(items)} records")
        time.sleep(1)                     # rule 2

print(f"\nCollected {len(all_rows)} records before deduplication")
if failures:
    print(f"⚠️  {len(failures)} page(s) failed and are MISSING: {failures}")
else:
    print("No failed pages.")

That last block matters more than it looks. If a page fails, the loop **says so**
and names it. Silence about a failure is worse than the failure — it is challenge
4 from Tuesday, where the program does not crash, it succeeds at producing
something incomplete.

In [ ]:
# DEMO — the same repository found by several keywords
df = pd.DataFrame(all_rows)
before = len(df)

dupes = df.duplicated(subset=["full_name"], keep="first").sum()
df = df.drop_duplicates(subset=["full_name"], keep="first").reset_index(drop=True)

print(f"Before deduplication : {before}")
print(f"Duplicates removed   : {dupes}")
print(f"Unique repositories  : {len(df)}")
print(f"\nShape: {df.shape[0]} rows x {df.shape[1]} columns")

Expect roughly a third of the records to vanish. That is not waste — it is the
same repository found by several of your keywords, which is what overlapping
search terms are supposed to do.

Look at *which* keywords overlap. The singular and plural forms will share most
of their results; `business analyst interview` will share far fewer. That is the
surface-versus-population distinction from section 4, now measured rather than
asserted.

Tuesday's version of this was the same job advertised by three agencies. Whatever
your source, **deduplicate on a stable identifier** — here `full_name`, which
GitHub guarantees is unique. Deduplicating on a title or a description will
merge things that only look alike.

---
# 🖐 YOUR TURN — Task 2 (10 minutes)

Build your own dataset.

1. Replace `MY_KEYWORDS` with two or three keywords of your own.
2. Add **two fields** of your choosing to `my_extract()`. Candidates:
   `size`, `watchers_count`, `has_wiki`, `archived`, `default_branch`,
   `open_issues_count`. Look at the raw JSON in section 7 to find more.
3. Run the collection and pass the checks.

Remember `or` for anything that might be null, and remember that lists must be
joined before they reach the DataFrame.

In [ ]:
# YOUR TURN — your keywords, your fields
MY_KEYWORDS = ["business analytics", "data analyst portfolio"]   # <-- change these


def my_extract(repo, keyword):
    licence = repo.get("license") or {}
    owner   = repo.get("owner") or {}
    row = {
        "full_name":        repo.get("full_name") or "",
        "description":      repo.get("description") or "",
        "stargazers_count": repo.get("stargazers_count", 0),
        "language":         repo.get("language") or "",
        "license_name":     licence.get("name") or "",
        "owner_type":       owner.get("type") or "",
        "topics":           "; ".join(repo.get("topics") or []),
        "html_url":         repo.get("html_url") or "",
        "search_keyword":   keyword,
        # TODO 1: add your first field here
        # TODO 2: add your second field here
    }
    return row


my_rows = []
for kw in MY_KEYWORDS:
    print(f"Searching: {kw!r}")
    for page in range(1, 2):                     # one page each keeps us quick
        items = fetch_page(kw, page, per_page=50)
        if not items:
            break
        my_rows.extend(my_extract(r, kw) for r in items)
        time.sleep(1)

my_df = pd.DataFrame(my_rows)
my_df["collected_at"] = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
my_df = my_df.drop_duplicates(subset=["full_name"]).reset_index(drop=True)

print(f"\n{len(my_df)} unique repositories, {len(my_df.columns)} columns")
my_df.head()

In [ ]:
# YOUR TURN — self-check
assert len(my_df) > 0, "No rows collected — check your keywords"
assert len(my_df.columns) >= 11, "You should have added two extra fields"
assert "collected_at" in my_df.columns, "Every row needs a collection date"
assert my_df["full_name"].is_unique, "Deduplication failed"
assert my_df["stargazers_count"].dtype != object, \
    "stargazers_count should be numeric, not text"
assert not my_df["topics"].apply(lambda x: isinstance(x, list)).any(), \
    "topics must be a string, not a list"

print("✓ All checks passed.")
print(f"  rows    : {len(my_df)}")
print(f"  columns : {list(my_df.columns)}")

The check on `stargazers_count` is the one that catches people. A column that
looks numeric but is stored as text will pass every eye test and fail every
calculation — Tuesday's price parsed as a string, in a different costume.

---
# 11. Provenance — what makes this evidence

A CSV on its own is a claim. A CSV somebody else can reproduce is evidence.
Rules 7 and 8 from Tuesday:

> **7.** Keep the raw response, not only the table.
> **8.** Log every run: date, URL, parameters, rows.

Three things get saved:

1. **The raw JSON**, exactly as it arrived — every page, not just the last one.
   If your parsing turns out to be wrong, you re-parse instead of re-collecting,
   and by then the web will have moved on. The file is several megabytes and full
   of fields you did not use. That is the point: you cannot know today which
   field you will need in November.
2. **The table**, with a `collected_at` stamp on every row.
3. **One log line per run**, appended to a file that grows across the semester.

This is what A1 asks for on 28 September, and it takes ten minutes.

In [ ]:
# DEMO — save all three
Path("data").mkdir(exist_ok=True)
Path("data/raw").mkdir(exist_ok=True)

stamp = datetime.now(timezone.utc)
stamp_file = stamp.strftime("%Y%m%d_%H%M%S")

# 1. every raw response, exactly as it arrived
raw_path = Path(f"data/raw/github_search_{stamp_file}.json")
raw_path.write_text(json.dumps(RAW_PAGES, ensure_ascii=False, indent=2), encoding="utf-8")

# 2. the table, stamped
df["collected_at"] = stamp.strftime("%Y-%m-%d %H:%M:%S UTC")
csv_path = Path(f"data/github_repos_{stamp_file}.csv")
df.to_csv(csv_path, index=False, encoding="utf-8-sig")   # utf-8-sig opens cleanly in Excel

# 3. the log line
log_path = Path("data/collection_log.csv")
log_entry = pd.DataFrame([{
    "collected_at":  stamp.strftime("%Y-%m-%d %H:%M:%S UTC"),
    "source":        API_URL,
    "keywords":      " | ".join(KEYWORDS),
    "sort":          SORT_BY,
    "pages":         MAX_PAGES,
    "per_page":      PER_PAGE,
    "rows_raw":      before,
    "rows_unique":   len(df),
    "failed_pages":  len(failures),
    "output_file":   csv_path.name,
}])
log_entry.to_csv(log_path, mode="a", header=not log_path.exists(),
                 index=False, encoding="utf-8-sig")

print(f"raw JSON : {raw_path}  ({len(RAW_PAGES)} pages, "
      f"{raw_path.stat().st_size / 1_048_576:.1f} MB)")
print(f"table    : {csv_path}  ({len(df)} rows)")
print(f"log      : {log_path}")

In [ ]:
# DEMO — three questions to ask of any dataset before you trust it
print("1. How complete is each column?")
missing = df.replace("", pd.NA).isna().sum()
print(missing[missing > 0].to_string() if missing.sum() else "   nothing missing")

print("\n2. Which keyword found what?")
print(df["search_keyword"].value_counts().to_string())

print("\n3. Do the types make sense?")
print(df.dtypes.to_string())

---
# 12. Takeaways

1. **An API is the front door.** When one exists, use it — you get typed fields, documented limits, and permission.
2. **The token is billed to you, not to the network.** That is why everyone needs their own — and why you clear it out of the notebook before you commit.
3. **A `200` is not a dataset.** `total_count` against `len(items)` is the first thing to check, every time.
4. **JSON arrives in three shapes** — flat, nested, list — and `.get()` with a default does not protect you from `null`.
5. **Save the raw response and log the run.** Ten minutes now; the difference between a claim and evidence in December.

### Before Tuesday

Tuesday is route 4 — HTML scraping, on university master's programme pages, where no API exists.

Bring three things:

1. **Your track's main source**, and the answer to one question: does it publish an API? Search *"[platform name] API documentation"*, or look for a "Developers" link in the site footer. If an API exists, rule 9 says use it.
2. **Its `robots.txt`**, read. Add `/robots.txt` to the site's home address.
3. **Fields** you intend to collect from a single record, written down.

If your track's source turns out to have an API, come and tell me — your collection just became much easier, and you can start on it before the lab.